<a href="https://colab.research.google.com/github/HemaShenoy/KAPTURECX/blob/main/finnetunee.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00


In [5]:
import json
import torch
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)

from peft import LoraConfig, get_peft_model


In [7]:
data = []

with open("cleaned_conversations.jsonl") as f:
    for line in f:
        convo = json.loads(line)

        for turn in convo["turns"]:
            if turn["role"] == "customer":
                prompt = turn["text"]

                response = "Sir EMI payment karna zaroori hai. Kya aap aaj payment kar sakte hain?"

                data.append({
                    "prompt": prompt,
                    "response": response
                })

print("Total training samples:", len(data))

Total training samples: 14


In [8]:
dataset = Dataset.from_list(data)

dataset


Dataset({
    features: ['prompt', 'response'],
    num_rows: 14
})

In [9]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded


In [10]:
def format_prompt(example):

    text = f"""
<user>
{example['prompt']}
</user>

<assistant>
{example['response']}
</assistant>
"""

    return {"text": text}


formatted = dataset.map(format_prompt)

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

In [13]:
def tokenize(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # add labels for training
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


tokenized_dataset = formatted.map(tokenize)

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

In [14]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("LoRA applied")

LoRA applied


In [15]:
training_args = TrainingArguments(
    output_dir="emi_agent_lora",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no"
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
